In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
dbutils.widgets.text('catalog','agmarknet')
dbutils.widgets.text('data source','Above_MSP_Prices')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data source')
print(catalog,data_source)

In [0]:
# define the tables
bronze_table = f"{catalog}.{bronze_schema}.dim_{data_source}"
silver_table = f"{catalog}.{silver_schema}.dim_{data_source}"
gold_dim_table = f"{catalog}.{gold_schema}.dim_{data_source}"
gold_fact_table = f"{catalog}.{gold_schema}.fact_{data_source}"

bronze_table, silver_table, gold_dim_table,gold_fact_table

In [0]:
today = datetime.today().strftime("%d-%m-%Y")
today

In [0]:
base_path = f's3://agmarknet-pc/AgmarkIncremental'
landing_path = f"{base_path}/AboveMSP/{today}/"
processed_path = f"{base_path}/AboveMSP/processed/{today}/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

In [0]:
#today = datetime.today().strftime("%d-%m-%Y")
file_path = f's3://agmarknet-pc/AgmarkIncremental/AboveMSP/{today}/*.csv'
print(file_path)

##preprocessing file before saving to bronze table

REad daily incrmental file from S3

In [0]:
df1 = spark.read.text(landing_path).select("*", "_metadata.file_name")

In [0]:

df1.select(F.col("file_name")).distinct().count()

In [0]:
###remove empty lines
df1 = df1.filter(F.col("value").isNotNull() & (F.trim(F.col("value")) != ""))

In [0]:
#add row number column
window = Window.partitionBy("file_name").orderBy(F.monotonically_increasing_id())
df1 = df1.withColumn("row_num",F.row_number().over(window))

In [0]:
df1.count()

In [0]:
## Add row type column to identify Header, metadata, commodity,  group, data rows
df = df1.withColumn("row_type", F.when(F.col("value").startswith("Group :"), "GROUP")
                   .when(F.col("value").startswith("State,"),"HEADER")
                   .when(F.col("value").startswith("Commodities reported Above (MSP) -"),"METADATA")
                   .when(F.col("value").rlike(r"^[A-Za-z][A-Za-z\s()&.-]*,"),"DATA")
                   .otherwise("COMMODITY")
)

###Extract the commodity group

In [0]:
df = (
    df.withColumn("Commodity_group",
                  F.when(F.col("value").startswith("Group :"),
                         F.regexp_extract("value",r"Group\s*:\s*(.*)",1)
                        ) 
                            )
)

##Forward-fill the value of commodity group

In [0]:
#Create the window per file
window_spec = (Window.partitionBy("file_name") 
    .orderBy("row_num").rowsBetween(Window.unboundedPreceding,Window.currentRow)
)

df = df.withColumn("Commodity_group",F.last("Commodity_group",ignorenulls=True).over(window_spec))
df.show()

In [0]:
#remove Meatadata rows 
df= df.filter(F.col("row_type").isin("DATA","HEADER","COMMODITY","GROUP"))
display(df)

##Identify Commodity rows

In [0]:
df = df.withColumn(
    "Commodity",
    F.when(
        F.col("row_type")== "COMMODITY" ,
        F.regexp_replace(F.col("value"),'^"|"$', "")  ## Remove double quotes from commodity name   
    )
)

df.show()

In [0]:
##Forward-fill the value of commodity 
w = Window.partitionBy("file_name").orderBy("row_num").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = df.withColumn("Commodity",F.last("Commodity", ignorenulls=True).over(w))
df.show()

In [0]:
#keep the data rows only
df = df.filter(F.col("row_type").isin("DATA"))
df.show()

In [0]:
# some of the Market center value contains comma, which shifts to the right column when saved as csv
#
df_csv = (
    df.withColumn("csv_line",
                  F.concat_ws(
                      ",",
                      F.concat(F.lit('"'),F.col("Commodity"),F.lit('"')),
                      F.concat(F.lit('"'),F.col("Commodity_group"),F.lit('"')),
                      F.col("value"),
                      F.concat(F.lit('"'),F.col("file_name"),F.lit('"'))
                  ))
)
display(df_csv)

In [0]:
# Writing csv_line as text to temp file
temp_path = "s3://agmarknet-pc/tmp1/23-07-2026/"
df_text = df_csv.select(F.col("csv_line"))
#df_text.show(10)
df_text.write.mode("overwrite").text(temp_path)

In [0]:
#Read the file as csv
#temp_path = "s3://agmarknet-pc/tmp1/"
final_df = spark.read.option("header","false") \
                     .option("quote",'"') \
                      .option("escape",'"') \
                      .csv(temp_path)

In [0]:
#Rename the columns
final_df = (final_df.toDF(
    "Commodity",
    "Commodity_group",
    'State', 'Market_Center', 'Variety', 'Grade', 'Date', 'Arrivals', 'Unit_of_Arrivals', 'Modal_Price', 'Unit_of_Price',
    "file_name"
)
)

In [0]:
ddf = final_df.filter(F.col("Market_Center").contains(","))
display(ddf)

In [0]:
print(final_df.columns)

In [0]:
final_df.count()

##Save the data to bronze table

In [0]:
### Save the df  to bronze schema
final_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("append") \
.saveAsTable(f'{catalog}.{bronze_schema}.dim_{data_source}')

##silver layer transformation

Staging table to process just the arrived incremenal data

In [0]:
final_df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeschema","true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [0]:
silver_df = spark.sql(f'select * from {catalog}.{bronze_schema}.staging_{data_source}')
display(silver_df)
silver_df.count()

In [0]:
#extract Commodity name and MSP price from Commodity column
silver_df = (
    silver_df.withColumn(
        "Commodity_Name",
        F.regexp_extract(
            "Commodity",
            r"^(.*)\s+\(MSP:",
            1
        )
    )
    .withColumn(
        "MSP_Price",
        F.regexp_extract(
            "Commodity",
            r"MSP:\s*Rs\.\s*([\d.]+)",
            1
        ).cast("decimal(10,2)")
    )
)

In [0]:
display(silver_df)

In [0]:
silver_df.count()

In [0]:
silver_df.printSchema()

In [0]:
#cast Arrival quantity and Modal price to double
silver_df = silver_df.withColumn("Arrivals", F.col("Arrivals").cast("double")) \
            .withColumn("Modal_Price", F.col("Modal_Price").cast("double"))

In [0]:
# analyze date formats before transformation
df_silver_dates = spark.sql(f'select * from {catalog}.{bronze_schema}.staging_{data_source}')

df_formats = (
    df_silver_dates.withColumn(
        "date_format",
        F.when(F.col("Date").rlike(r"^\d{2}-\d{2}-\d{4}$"), "dd-MM-yyyy")
         .when(F.col("Date").rlike(r"^\d{2}/\d{2}/\d{4}$"), "dd/MM/yyyy")
         .when(F.col("Date").rlike(r"^\d{4}-\d{2}-\d{2}$"), "yyyy-MM-dd")
         .when(F.col("Date").rlike(r"^\d{4}/\d{2}/\d{2}$"), "yyyy/MM/dd")
         .when(F.col("Date").rlike(r"^\d{2}-[A-Za-z]{3}-\d{4}$"), "dd-MMM-yyyy")
         .otherwise("Unknown")
    )
)

df_formats.groupBy("date_format").count().show(truncate=False)

In [0]:
df_formats.filter(F.col("date_format")=="Unknown").show()

In [0]:
silver_df = silver_df.withColumn("Date",F.to_date("Date","dd-MM-yyyy"))

In [0]:
silver_df.printSchema()

In [0]:
silver_df = silver_df.withColumn("Arrivals",F.round(F.col("Arrivals"),2).cast("decimal(10,2)"))

###Save/Append the latest data to silver table

In [0]:
spark.sql(f"select count(*) from {catalog}.{silver_schema}.dim_{data_source} where Date = '2026-07-23' ").show()

In [0]:
silver_df.count()

In [0]:
185696+488

In [0]:
silver_delta = DeltaTable.forName(spark,silver_table)
silver_delta.alias("silver").merge(silver_df.alias("new"),
                                   """
                                   silver.Date= new.Date AND
                                   silver.Commodity_Name =new.Commodity_Name AND
                                   silver.Commodity_group=new.Commodity_group
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Staging silver table to process just the arrived incremenal data

In [0]:
### Save the df  to silver schema
silver_df.write \
.format("delta") \
.option("overwriteSchema","true") \
.option("delta.enableChangeDataFeed", "true") \
.option("mergeSchema", "true") \
.mode("overwrite") \
.saveAsTable(f'{catalog}.{silver_schema}.staging_{data_source}')

###Gold layer

In [0]:
gold_df = spark.sql(f'select * from {catalog}.{silver_schema}.staging_{data_source}')
gold_df = gold_df.withColumn("Msp_Status",F.lit("Above MSP"))
display(gold_df)

###writing/append new data to gold table

In [0]:
gold_delta = DeltaTable.forName(spark,gold_dim_table)
gold_delta.alias("gold").merge(gold_df.alias("new"),
                                   """
                                   gold.Date= new.Date AND
                                   gold.Commodity_Name =new.Commodity_Name AND
                                   gold.Commodity_group=new.Commodity_group
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
gold_df.count()

%md
###Creating msp table with facts 

In [0]:
comm_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_commodity')
comm_grp_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_commodity_group')
market_df = spark.sql(f'select * from {catalog}.{gold_schema}.dim_market')

In [0]:
# join comodity df and gold df to add commodity code
gold_df = gold_df.alias("b").join(
   comm_df.alias("c"),
   (F.col("b.Commodity_Name") == F.col("c.Commodity")) &
   (F.col("b.Variety") == F.col("c.Variety")) &
   (F.col("b.Grade") == F.col("c.Grade")),
   "left"
).select(
   "c.Commodity_Code",
   "b.*"
)

In [0]:
gold_df.count()

In [0]:
# join comodity group df and gold df to add commodity group id
# 
gold_df1 = gold_df.alias("b").join(
   comm_grp_df.alias("c"),
   (F.col("b.Commodity_group") == F.col("c.Commodity_Group_Name")),
   "left"
).select(
   "c.Commodity_Group_Id",
   "b.*"
)
display(gold_df1)

In [0]:
# join goldf1 and market df to get the market code

fact_abmsp_df = (
    gold_df1.alias("g")
    .join(
        market_df.alias("m"),
        (F.col("g.Market_Center") == F.col("m.market_name")) & (F.col("g.State") == F.col("m.state_name")),
        "left")
    .select("g.Commodity_Group_Id",
            "g.Commodity_Code",
            "g.MSP_Price",
            "m.market_id",
            "m.district_id",
            "m.state_id",
            "g.Date",
            "g.Arrivals",
            "g.Unit_of_Arrivals",
            "g.Modal_Price",
            "g.Unit_of_Price",
            "g.Msp_Status"
    )
)

####Append fact df to fact above msp table

In [0]:
gold_delta1 = DeltaTable.forName(spark,gold_fact_table)
gold_delta1.alias("gold").merge(fact_abmsp_df.alias("new"),
                                   """
                                   gold.Date= new.Date AND
                                   gold.Commodity_Code =new.Commodity_Code AND
                                   gold.Commodity_Group_Id=new.Commodity_Group_Id AND
                                   gold.market_id=new.market_id AND
                                   gold.state_id=new.state_id
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

##Append fact msp data to main fact msp table

In [0]:
fact_delta = DeltaTable.forName(spark,fact_table)
fact_delta.alias("gold").merge(fact_abmsp_df.alias("new"),
                                   """
                                   gold.Date= new.Date AND
                                   gold.Commodity_Code =new.Commodity_Code AND
                                   gold.Commodity_Group_Id=new.Commodity_Group_Id AND
                                   gold.market_id=new.market_id AND
                                   gold.state_id=new.state_id AND
                                   gold.Msp_Status=new.Msp_Status
                                   """
                                   ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

###Clean up


In [0]:
%sql
drop table agmarknet.bronze.staging_above_msp_prices

In [0]:
%sql
drop table agmarknet.silver.staging_above_msp_prices